# Exercise: Autoregressive Text Generation with Temperature Sampling

Welcome! In this exercise, you'll implement one of the most exciting capabilities of a language model: generating new text. You will extend your `MiniGPT` model with a `generate` method that can write new text, one token at a time, based on a starting prompt.

This is a key step in understanding how models like ChatGPT can produce coherent and creative responses. Let's get started!

### In this exercise you will:
- Implement the core autoregressive loop that generates text one token at a time.
- Incorporate temperature sampling to control the creativity and randomness of the model's output.
- Build and test a complete `generate` method for your `MiniGPT` model.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Pre-built code ---
# You do not need to modify this part.

class GPTConfig:
    """A simple configuration class for our model."""
    def __init__(self, vocab_size=65, block_size=128, **kwargs):
        self.vocab_size = vocab_size
        self.block_size = block_size
        for k, v in kwargs.items():
            setattr(self, k, v)

class MiniGPT(nn.Module):
    """
    A simplified GPT model for this exercise.
    You will be implementing the `generate` method for this class.
    """
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        # A simple model architecture
        self.embedding = nn.Embedding(config.vocab_size, 64)
        self.transformer_block = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
        )
        self.lm_head = nn.Linear(64, config.vocab_size)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """
        A dummy forward pass that returns logits.
        In a real model, this would involve transformer blocks.
        """
        B, T = idx.shape
        # Ensure sequence length doesn't exceed the model's block size
        assert T <= self.config.block_size, f"Sequence length {T} exceeds block size {self.config.block_size}"
        
        # Get token embeddings
        x = self.embedding(idx) # (B, T, embedding_dim)
        # Pass through a simple transformation
        x = self.transformer_block(x) # (B, T, embedding_dim)
        # Project to vocabulary size to get logits
        logits = self.lm_head(x) # (B, T, vocab_size)
        return logits

## Exercise 1: Implement Autoregressive Generation

Your first task is to implement the main loop for text generation. The model works *autoregressively*, meaning it generates one token, adds it to the input sequence, and then uses that new sequence to generate the next token, and so on.

**Your task:** Implement the `generate` method below.

**Instructions:**
1.  Start with the initial `prompt_tokens` as your context. We'll call this context `idx`.
2.  Loop for `max_new_tokens` iterations to generate that many new tokens.
3.  Inside the loop:
    a.  **Crop context:** The model can only process sequences up to `config.block_size`. If your current context `idx` is longer than that, trim it so you're only using the last `block_size` tokens as input.
    b.  **Get logits:** Pass the (potentially cropped) context to the model's `forward` method to get the predictions (`logits`).
    c.  **Focus on the last token:** We only need the logits for the very next token, which are the predictions corresponding to the *last* token in the input sequence. Slice the `logits` tensor to get just these.
    d.  **Get probabilities:** Apply a `softmax` function to the logits to convert them into probabilities.
    e.  **Sample the next token:** Use `torch.multinomial` to sample a single token from the probability distribution you just created. This will be your newly generated token.
    f.  **Append and continue:** Append the sampled token to your context `idx`. This new, longer context will be used in the next iteration of the loop.
4.  After the loop finishes, return the complete sequence of tokens (`idx`).

**Hint:** `torch.multinomial` is the perfect tool for sampling from a probability distribution. It takes probabilities as input and gives you the index of the sampled item.

In [ ]:
def generate(self, prompt_tokens: torch.Tensor, max_new_tokens: int, temperature: float = 1.0) -> torch.Tensor:
    """
    Generates new tokens autoregressively based on a prompt.

    Args:
        prompt_tokens (torch.Tensor): The initial sequence of tokens (B, T).
        max_new_tokens (int): The number of new tokens to generate.
        temperature (float): A factor to scale the logits, controlling randomness.
                             A higher value (e.g., > 1.0) makes the output more random,
                             while a lower value (e.g., < 1.0) makes it more deterministic.

    Returns:
        torch.Tensor: The generated sequence of tokens, including the prompt (B, T + max_new_tokens).
    """
    # Make sure we're in evaluation mode
    self.eval()
    # The context starts with the prompt
    idx = prompt_tokens
    
    # Generate tokens one by one
    for _ in range(max_new_tokens):
        ### START CODE HERE ###
        # Step 3a: Crop the context if it exceeds block_size
        idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
        
        # Step 3b: Get the logits from the model
        logits = self(idx_cond) # self(idx) is equivalent to self.forward(idx)
        
        # Step 3c: Focus only on the logits for the last time step
        logits = logits[:, -1, :] # Becomes (B, C)
        
        # Step 3d (Exercise 2): Apply temperature scaling
        logits = logits / temperature

        # Step 3e: Apply softmax to get probabilities
        probs = F.softmax(logits, dim=-1) # (B, C)
        
        # Step 3f: Sample from the distribution
        idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
        
        # Step 3g: Append the sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        ### END CODE HERE ###

    return idx

# We will attach this method to our MiniGPT class
MiniGPT.generate = generate

In [ ]:
# --- Test cell for Exercise 1 ---
# Note: Since the prompt for Exercise 1 asks you to implement the full logic including temperature,
# this test cell will validate your complete implementation.

print("Running tests for the generate method...")

# Use a fixed seed for reproducibility
torch.manual_seed(42)

# 1. Setup the model and inputs
config = GPTConfig(vocab_size=50, block_size=128)
model = MiniGPT(config)
prompt = torch.tensor([[1, 2, 3]], dtype=torch.long) # Batch size of 1, sequence length of 3

# 2. Test Case: Low temperature (greedy decoding)
# With a very low temperature, the model should always pick the token with the highest logit.
low_temp_tokens = model.generate(prompt, max_new_tokens=5, temperature=0.01)
expected_low_temp = torch.tensor([[1, 2, 3, 3, 3, 3, 3, 3]])
assert torch.equal(low_temp_tokens, expected_low_temp), f"Low temperature test failed! Expected {expected_low_temp}, but got {low_temp_tokens}"
print("✅ Low temperature test passed!")

# 3. Test Case: Standard temperature
# With a fixed seed, the "random" sampling becomes deterministic.
std_temp_tokens = model.generate(prompt, max_new_tokens=5, temperature=1.0)
expected_std_temp = torch.tensor([[ 1,  2,  3, 39, 39, 39, 39, 39]])
assert torch.equal(std_temp_tokens, expected_std_temp), f"Standard temperature test failed! Expected {expected_std_temp}, but got {std_temp_tokens}"
print("✅ Standard temperature test passed!")

# 4. Test Case: High temperature (more random)
high_temp_tokens = model.generate(prompt, max_new_tokens=5, temperature=2.5)
expected_high_temp = torch.tensor([[ 1,  2,  3, 31, 15, 27,  6,  9]])
assert torch.equal(high_temp_tokens, expected_high_temp), f"High temperature test failed! Expected {expected_high_temp}, but got {high_temp_tokens}"
print("✅ High temperature test passed!")

# 5. Test Case: Context cropping
# Create a prompt longer than the block size to test cropping logic.
long_prompt = torch.arange(0, 150).unsqueeze(0) # Shape (1, 150)
config_short = GPTConfig(vocab_size=50, block_size=100)
model_short = MiniGPT(config_short)
torch.manual_seed(42) # Reset seed for this specific test
cropped_tokens = model_short.generate(long_prompt, max_new_tokens=2, temperature=1.0)
assert cropped_tokens.shape == (1, 152), f"Cropping test failed! Expected shape (1, 152), got {cropped_tokens.shape}"
print("✅ Context cropping test passed!")

print("\n🎉 All tests passed! Great job on implementing the generate method!")

## You've done it!

Congratulations! You have successfully implemented a `generate` method that can produce new text autoregressively. You also integrated temperature scaling, a powerful technique for controlling the model's output.

You now have a solid understanding of:
-   The step-by-step process of autoregressive generation.
-   How to use `softmax` and `torch.multinomial` to sample from the model's predictions.
-   The role of `temperature` in balancing between deterministic and creative generation.

## Optional Challenge: `top_k` Sampling

For an extra challenge, you can extend your `generate` method to include `top_k` sampling. This is another popular technique to control generation quality.

**The `top_k` logic:**
1.  After getting the logits for the last time step, identify the `k` highest logit values.
2.  Create a new tensor where all logits *not* in the top `k` are set to a very small number (like negative infinity). This effectively removes them from consideration.
3.  Proceed with the temperature scaling and `softmax` as before, but on these modified logits.

You can modify your `generate` function to accept an optional `top_k` argument and add this logic.

---
`generate(self, ..., top_k: int = None)`

Inside the loop, after you get `logits`:
```python
if top_k is not None:
    # Your top_k implementation here
    # 1. Get the top k values and their indices
    # 2. Create a new tensor of -inf
    # 3. Place the top k logits back into the new tensor
    # 4. Replace the original logits with this new tensor
```
This is a great way to further improve your text generation skills!

In [ ]:
# --- Integration Test: Let's see it in action! ---
# This cell demonstrates how your implemented `generate` method can be used
# to create text with a simple character-level "tokenizer".

print("Running a demonstration of text generation...")

# 1. Create a simple vocabulary and tokenizer
text = "hello world